# Baseline
Den övergripande explorativa dataanalysen genomförs separat i projektets EDA 01_exploration. Den här notebooken fokuserar på att bygga och testa projektets baseline för kompetensidentifiering.

In [1]:
# läser in annonser:
import json
import pandas as pd

with open("../data/raw_ads_2511.json", "r", encoding="utf-8") as file:
    ads = json.load(file)

df = pd.DataFrame(ads)

In [2]:
# liten EDA som är relevant för baseline:
df[["id", "title", "published", "region", "description"]].head()

,id,title,published,region,description
0,29427644,Domänarkitekt,2025-01-31T09:54:18,Stockholms län,Vill du driva teknisk förändring i en av de vi...
1,29403435,Industriell verksamhetsutvecklare inom IT och ...,2025-01-22T17:38:54,Västra Götalands län,"Bli en del av Prevas, ett innovativt företag i..."
2,29403435,Industriell verksamhetsutvecklare inom IT och ...,2025-01-22T17:38:54,Västra Götalands län,"Bli en del av Prevas, ett innovativt företag i..."
3,29403435,Industriell verksamhetsutvecklare inom IT och ...,2025-01-22T17:38:54,Västra Götalands län,"Bli en del av Prevas, ett innovativt företag i..."
4,29403435,Industriell verksamhetsutvecklare inom IT och ...,2025-01-22T17:38:54,Västra Götalands län,"Bli en del av Prevas, ett innovativt företag i..."


In [3]:
# dubbelkollar saknade värden:
df["description"].isna().sum()

np.int64(0)

In [4]:
# dubbelkollar id-dubbletter:
df["id"].duplicated().sum()

np.int64(4574)

Baseline utvecklas tillfälligt på rådata, den slutliga analysen ska köras på gruppens rensade JSON-fil.

In [5]:
# läser in kompetenserna för SSYK 2511
with open("../data/skills_2511.json", "r", encoding="utf-8") as file:
    skills_data = json.load(file)

In [6]:
print("\nStruktur:",type(skills_data))
print("----------------------------")
skills_data[0]


Struktur: <class 'list'>
----------------------------


{'skill_id': '13ad_dCC_Tn6',
 'skill': 'OOS, styr- och utvecklingsmodell',
 'headlines': ['Styr- och utvecklingsmodeller']}

In [7]:
# bygger en lista som baseline kan matcha mot:
skill_names = [
    item["skill"]
    for item in skills_data
]

print("\nAntal kompetenser:", len(skill_names))
print("----------------------------")

print("\nExempel på 20 kompetenser:")
skills_df = pd.DataFrame(skills_data)
print(skills_df.head(20))
print("----------------------------")


Antal kompetenser: 204
----------------------------

Exempel på 20 kompetenser:
        skill_id                                              skill  \
0   13ad_dCC_Tn6                   OOS, styr- och utvecklingsmodell   
1   16ZN_3Zp_bYV                                Odoo, molnplattform   
2   19PK_uxU_Wzy                              Bower, pakethanterare   
3   1CmG_8hR_igc                                   Bluetooth-teknik   
4   1QYQ_Xka_TQ5      Microsoft Autopilot, systemutvecklingsverktyg   
5   2CXh_FUJ_vTH  Office SharePoint Server (MOSS), applikationsp...   
6   2rqG_WLp_em4                   Ledningssystem (ISO 9000-serien)   
7   2vCi_hQz_Y9o                                         WAP-teknik   
8   39C3_4Tm_3ud                       Asset Information Management   
9   3H7m_pt8_7b2                HTTP/HTTPS, kommunikationsprotokoll   
10  3JPR_oR7_UUS      Sun Java Enterprise System, webbserverprogram   
11  3KME_Qwm_D4V                                IBM Integration Bus

In [8]:
# Baseline funktionen:
def baseline_extract_skills(text, skills):
    found_skills = []

    text = text.lower()

    for skill in skills:
        if skill.lower() in text:
            found_skills.append(skill)

    return found_skills

In [9]:
# testar baseline på några annonser:
test_df = df.head(20).copy()

test_df["baseline_skills"] = test_df["description"].apply(
    lambda text: baseline_extract_skills(text, skill_names)
)

In [10]:
test_df[
    ["title", "baseline_skills"]
]

,title,baseline_skills
0,Domänarkitekt,"[Agila arbetsmetoder, REST]"
1,Industriell verksamhetsutvecklare inom IT och ...,[]
2,Industriell verksamhetsutvecklare inom IT och ...,[]
3,Industriell verksamhetsutvecklare inom IT och ...,[]
4,Industriell verksamhetsutvecklare inom IT och ...,[]
5,Managing Application Consultant,[REST]
6,IT-konsult,[]
7,Förvaltningsledare IT till Nexer,[]
8,Applikationskonsult till Nexer,[]
9,"Cybersecurity Solution Architect, Mobility",[]


Baseline fungerar tekniskt, men många annonser får inga träffar. Det kan bero på att:
- taxonomin inte innehåller exakt samma formulering som annonsen
- annonsen skriver en förkortning, t.ex. ML istället för Machine Learning
- kompetensen uttrycks på engelska/svenska på annat sätt
- kompetensen finns implicit men inte som exakt fras
- SSYK 2511-listan inte täcker exakt allt som står i varje annons

In [ ]:
# Undersöker tomma träffar :
empty_matches = test_df[
    test_df["baseline_skills"].apply(len) == 0
]

sample_errors = empty_matches.head(5)

In [ ]:
# Annonser med empty matches:
sample_errors[
    ["title", "description"]
]

,title,description
1,Industriell verksamhetsutvecklare inom IT och ...,"Bli en del av Prevas, ett innovativt företag i..."
2,Industriell verksamhetsutvecklare inom IT och ...,"Bli en del av Prevas, ett innovativt företag i..."
3,Industriell verksamhetsutvecklare inom IT och ...,"Bli en del av Prevas, ett innovativt företag i..."
4,Industriell verksamhetsutvecklare inom IT och ...,"Bli en del av Prevas, ett innovativt företag i..."
6,IT-konsult,Om jobbet\nCapgemini strävar efter att vara en...


In [ ]:
# Exempel på annonserna med empty matches som vi kan granska manuellt, felanalys:
for i, row in sample_errors.iterrows():
    print("\n" + "=" * 80)
    print("TITLE:", row["title"])
    print("BASELINE:", row["baseline_skills"])
    print("\nDESCRIPTION:")
    print(row["description"])


TITLE: Industriell verksamhetsutvecklare inom IT och system
BASELINE: []

DESCRIPTION:
Bli en del av Prevas, ett innovativt företag inom IT-tjänster. Som analytisk affärskonsult får du möjlighet att utveckla kreativa och teknikbaserade lösningar inom olika områden. Utmana konventioner och driva innovation tillsammans med oss. Ditt jobb på Prevas

På Prevas jobbar man ofta i projektform och dessa kan variera i både inriktning och storlek. Arbetet utgår från kontoret i centrala Göteborg men kan också utföras hemifrån eller hos kund. Du kommer jobba med industriella processer, där du agerar rådgivare och visar vägen för kunderna i deras effektivisering- och förbättringsarbete. Här mappar du kundens nuläge och börläge och hjälper dem att identifiera och implementera resan däremellan. Du projektleder och driver beslutade insatser – alltifrån lean initiativ till digitalisering- eller automatisering. I rollen är det din metodik och ditt stora driv som blir ditt främsta redskap. På Prevas bli

Baseline gav inga träffar. Vid manuell granskning innehöll annonsen främst generella beskrivningar av IT-arbete, projektledning och systemutveckling, men inga tydliga kompetenser som matchade vår skill-lista exakt. Resultatet bedömder vi därför som rimligt.

In [15]:
# sammanställer resultatet på hela datasetet och inte endast 20 annonser:

df["baseline_skills"] = df["description"].apply(
    lambda text: baseline_extract_skills(text, skill_names)
)

all_skills = df["baseline_skills"].explode()

skill_counts = all_skills.value_counts()

skill_counts.head(20)

baseline_skills
REST                       2717
Informationsarkitektur      361
Förändringsledning          335
Datamodellering             298
Masterdata                  228
Informationsmodellering     200
Systemförvaltning           181
Informatik                  181
Agila arbetsmetoder         115
Verksamhetsanalys            89
API-design                   64
Molnteknik                   29
Nätverksteknik               29
Design Thinking              27
Prediktiv analys             25
Projektledning, IT           23
Datawarehouse                23
Webbutveckling               22
Budgetansvar                 22
Nätverksdesign               15
Name: count, dtype: int64

`REST = 2717` betyder att baseline hittade termen REST i 2717 annonser, men vad är REST för något ?

In [ ]:
# Undersöker i vilka annonser förekommer REST: 
rest_ads = df[
    df["baseline_skills"].apply(lambda x: "REST" in x)
]

rest_ads[["title", "description"]].head()

,title,description
0,Domänarkitekt,Vill du driva teknisk förändring i en av de vi...
5,Managing Application Consultant,"Together with a strong, multinational Capgemin..."
11,AI Engineer,We offer a supportive work environment that en...
13,Junior Data and Financial Analyst till Modity,Vill du ta nästa steg i din karriär i en dynam...
14,Senior C#/.NET Developer -11452,Veritaz is a leading IT staffing solutions pro...


In [ ]:
# Kollar upp vart exakt fångar baseline ordet REST:
text = df.loc[0, "description"].lower()

position = text.find("rest")

print(position)
print(text[position-50:position+50])

3158
assa i rollen som domänarkitekt ser vi att du är prestigelös, engagerad och självständigt driver dit


In [18]:
import re

text = df.loc[0, "description"].lower()

for match in re.finditer("rest", text):
    start = match.start()

    print(
        text[max(0, start-40):start+40]
    )
    print("--------------------")

len som domänarkitekt ser vi att du är prestigelös, engagerad och självständigt 
--------------------


In [19]:
# Förbättrad baseline: 
import re

def improved_baseline_extract_skills(text, skills):
    found_skills = []

    text = text.lower()

    for skill in skills:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [20]:
test_df["improved_baseline_skills"] = test_df["description"].apply(
    lambda text: improved_baseline_extract_skills(text, skill_names)
)

In [21]:
test_df[
    [
        "title",
        "baseline_skills",
        "improved_baseline_skills"
    ]
]

,title,baseline_skills,improved_baseline_skills
0,Domänarkitekt,"[Agila arbetsmetoder, REST]",[Agila arbetsmetoder]
1,Industriell verksamhetsutvecklare inom IT och ...,[],[]
2,Industriell verksamhetsutvecklare inom IT och ...,[],[]
3,Industriell verksamhetsutvecklare inom IT och ...,[],[]
4,Industriell verksamhetsutvecklare inom IT och ...,[],[]
5,Managing Application Consultant,[REST],[]
6,IT-konsult,[],[]
7,Förvaltningsledare IT till Nexer,[],[]
8,Applikationskonsult till Nexer,[],[]
9,"Cybersecurity Solution Architect, Mobility",[],[]


In [22]:
df["improved_baseline_skills"] = df["description"].apply(
    lambda text: improved_baseline_extract_skills(text, skill_names)
)

In [23]:
all_improved_skills = df["improved_baseline_skills"].explode()

improved_skill_counts = all_improved_skills.value_counts()

improved_skill_counts.head(20)

improved_baseline_skills
Informationsarkitektur     361
Förändringsledning         335
Datamodellering            291
REST                       274
Informationsmodellering    200
Masterdata                 189
Informatik                 143
Systemförvaltning          134
Agila arbetsmetoder        115
Verksamhetsanalys           79
API-design                  64
Design Thinking             27
Prediktiv analys            25
Projektledning, IT          23
Webbutveckling              22
Budgetansvar                22
Datawarehouse               20
Nätverksteknik              19
Molnteknik                  16
Nätverksdesign              15
Name: count, dtype: int64

In [24]:
comparison = pd.DataFrame({
    "original_baseline": skill_counts,
    "improved_baseline": improved_skill_counts
}).fillna(0)

comparison.head(20)

,original_baseline,improved_baseline
5G-teknik,2,2.0
API-design,64,64.0
Agila arbetsmetoder,115,115.0
Automationsteknik,4,4.0
Budgetansvar,22,22.0
Configuration Management/CM,1,0.0
Databasdesign,7,7.0
Datamodellering,298,291.0
Datawarehouse,23,20.0
Design Thinking,27,27.0


Den ursprungliga baselinen använde enkel substring-matchning, vilket gav falska positiva träffar. Ett tydligt exempel var kompetensen REST, som bland annat matchades i ordet "prestigelös". Efter att matchningen förbättrades med reguljära uttryck och ordgränser minskade REST från 2717 till 274 träffar, medan flera andra kompetenser behöll samma antal träffar. Detta tyder på att den förbättrade baselinen minskar antalet falska positiva matchningar.

### Baseline bygger på exakt strängmatchning mellan kompetenser från SSYK 2511-taxonomin och jobbannonsernas beskrivningar. Metoden fungerar som en enkel referensmodell men är begränsad eftersom den kan missa förkortningar, synonymer och alternativa formuleringar. Resultatet kommer därför senare att jämföras med projektets NLP-metoder.